# Лекция 2. Rest API

## API

API (Application Programming Interface) - интерфейс програмного приложения приложения. 

Скорее всего, Вы уже много раз сталкивались с интерфейсами как с концепцией в программировании:

In [18]:
import time

time.__dict__

{'__name__': 'time',
 '__doc__': 'This module provides various functions to manipulate time values.\n\nThere are two standard representations of time.  One is the number\nof seconds since the Epoch, in UTC (a.k.a. GMT).  It may be an integer\nor a floating-point number (to represent fractions of seconds).\nThe epoch is the point where the time starts, the return value of time.gmtime(0).\nIt is January 1, 1970, 00:00:00 (UTC) on all platforms.\n\nThe other representation is a tuple of 9 integers giving local time.\nThe tuple items are:\n  year (including century, e.g. 1998)\n  month (1-12)\n  day (1-31)\n  hours (0-23)\n  minutes (0-59)\n  seconds (0-59)\n  weekday (0-6, Monday is 0)\n  Julian day (day in the year, 1-366)\n  DST (Daylight Savings Time) flag (-1, 0 or 1)\nIf the DST flag is 0, the time is given in the regular time zone;\nif it is 1, the time is given in the DST time zone;\nif it is -1, mktime() should guess based on the date and time.\n',
 '__package__': '',
 '__loader__

In [19]:
for k, v in time.__dict__.items():
    if hasattr(v, "__call__"):
        print(k, v)

__loader__ <class '_frozen_importlib.BuiltinImporter'>
time <built-in function time>
time_ns <built-in function time_ns>
clock_gettime <built-in function clock_gettime>
clock_gettime_ns <built-in function clock_gettime_ns>
clock_settime <built-in function clock_settime>
clock_settime_ns <built-in function clock_settime_ns>
clock_getres <built-in function clock_getres>
sleep <built-in function sleep>
gmtime <built-in function gmtime>
localtime <built-in function localtime>
asctime <built-in function asctime>
ctime <built-in function ctime>
mktime <built-in function mktime>
strftime <built-in function strftime>
strptime <built-in function strptime>
tzset <built-in function tzset>
monotonic <built-in function monotonic>
monotonic_ns <built-in function monotonic_ns>
process_time <built-in function process_time>
process_time_ns <built-in function process_time_ns>
thread_time <built-in function thread_time>
thread_time_ns <built-in function thread_time_ns>
perf_counter <built-in function per

In [20]:
print(time.sleep.__doc__)

sleep(seconds)

Delay execution for a given number of seconds.  The argument may be
a floating-point number for subsecond precision.


Поверхностно мы смогли познакомиться с интерфейсом библиотеки time, можем увидеть какие есть функции у библиотеки - это и есть интерфейс. Теперь мы хотим понять что такое интерфейс приложения. Фактически - это то, какие функции вовне предлагает наше приложение.

В реальной жизни клиент и сервер должны договориться о том, какие данные и как будут передаваться.

## HTTP

HTTP (HyperText Transfer Protocol) — протокол передачи гипертекста. 
Изначально создавался для передачи HTML-документов, но сейчас используется 
повсеместно как основа обмена данными в вебе.

HTTP — это протокол **клиент-серверного** взаимодействия, где клиент отправляет 
**запрос (request)**, а сервер возвращает **ответ (response)**.

### Структура HTTP-запроса

1. **Стартовая строка** — метод, URL, версия протокола (например, `GET /api/users HTTP/1.1`)
2. **Заголовки (headers)** — мета-информация (`Content-Type`, `Authorization`, и т.д.)
3. **Тело (body)** — передаваемые данные (не у всех методов)

### HTTP-методы

| Метод | Назначение |
|-------|------------|
| GET | Получить ресурс |
| POST | Создать новый ресурс |
| PUT | Полностью заменить ресурс |
| PATCH | Частично обновить ресурс |
| DELETE | Удалить ресурс |

### Коды ответов (status codes)

- **1xx** — Информационные (100 Continue)
- **2xx** — Успех (200 OK, 201 Created, 204 No Content)
- **3xx** — Перенаправление (301 Moved Permanently, 304 Not Modified)
- **4xx** — Ошибка клиента (400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found)
- **5xx** — Ошибка сервера (500 Internal Server Error, 502 Bad Gateway, 503 Service Unavailable)


## Принципы REST

REST (Representational State Transfer) — архитектурный стиль для проектирования 
сетевых API, предложенный Роем Филдингом в 2000 году.

На практике под REST API обычно понимают HTTP-сервис, который следует 
определённым принципам:

### 1. Resource (Ресурс)
Всё, с чем работает API — это **ресурсы**. Ресурсом может быть пользователь, 
заказ, товар, статья. Каждый ресурс идентифицируется **URI** (например, `/users`, `/orders/42`).

### 2. HTTP-методы как CRUD
Операции над ресурсами отображаются на HTTP-методы:

| Действие | CRUD | HTTP-метод | Пример |
|----------|------|------------|--------|
| Получить список | Read | GET | `GET /users` |
| Получить один | Read | GET | `GET /users/1` |
| Создать | Create | POST | `POST /users` |
| Заменить | Update | PUT | `PUT /users/1` |
| Обновить частично | Update | PATCH | `PATCH /users/1` |
| Удалить | Delete | DELETE | `DELETE /users/1` |

### 3. Stateless (Отсутствие состояния)
Каждый запрос от клиента к серверу должен содержать всю информацию, 
необходимую для его обработки. Сервер **не хранит** состояние клиента 
между запросами (нет сессий на стороне сервера). Авторизация передаётся 
с каждым запросом (обычно через заголовок `Authorization`).

### 4. Единообразие интерфейса (Uniform Interface)
Используются стандартные HTTP-методы, коды ответов, форматы данных (обычно JSON). 
Имена ресурсов — существительные во множественном числе (`/users`, а не `/getUsers`).

### Пример RESTful-запроса

Допустим, мы хотим получить пользователя с id=42:
```
GET /users/42 HTTP/1.1
Host: api.example.com
Authorization: Bearer <токен>
Accept: application/json
```

Ответ:
```
HTTP/1.1 200 OK
Content-Type: application/json

{
  "id": 42,
  "name": "Иван Иванов",
  "email": "ivan@example.com"
}
```

## Python-фреймворки для веб-API

### FastAPI
Современный, быстрый (один из самых производительных) фреймворк. 
Основан на **type hints** Python, автоматически генерирует 
OpenAPI/Swagger-документацию. Поддерживает асинхронность из коробки.
Отлично подходит для микросервисов и новых проектов.

### Flask
Лёгкий микро-фреймворк с минимальным порогом входа. 
Хорош для небольших сервисов и прототипов. Функциональность 
расширяется через плагины (Flask-SQLAlchemy, Flask-Migrate и т.д.).

### Django + Django REST Framework (DRF)
Полноценный фреймворк "всё включено". DRF — стандартное решение для 
создания REST API в экосистеме Django. Содержит ORM, админку, 
миграции, сериализаторы. Подходит для крупных монолитных приложений.

### Aiohttp
Асинхронный фреймворк как для сервера, так и для клиента. 
Предоставляет низкоуровневый контроль над HTTP.

---

**Сравнение:** Flask — для простоты и прототипов, FastAPI — для современного 
API с документацией и типами, Django — для больших проектов с богатой экосистемой.